## Exercise 3: Install Libraries Notebook-Scoped

At HealthBridge Analytics, different teams sometimes need different versions of the same library. Rather than creating a new cluster for every variation, you can install libraries **notebook-scoped** — they apply only to the current notebook session and do not affect other users or notebooks on the same compute resource.

In this exercise, you install the `faker` library notebook-scoped and verify it works correctly.

[faker](https://faker.readthedocs.io/) is a Python library that generates realistic fake data — such as names, addresses, phone numbers, dates, and more — for use in testing, prototyping, and seeding databases. It supports many locales and data categories, making it easy to produce large volumes of plausible synthetic data without using real personal information.

### Task 3.1 — Install the `faker` library notebook-scoped

Your team relies on `faker` to generate synthetic patient records for pipeline testing. Install it as a notebook-scoped library so the package is available to this notebook only and doesn't affect other users sharing the same compute.

> 🤖 **Genie Code tip:** Open Genie Code panel (click the ![assistant-icon](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Allfiles/media/genie-code.svg) icon on the cell) and use the prompt below to get started:
> *"How do I install a Python package notebook-scoped in Databricks using a magic command?"*

**Hint:** Use a `%pip` magic command to install the package. Pinning an exact version (e.g., `faker==40.8.0`) is a good practice for reproducibility.

In [0]:
%pip install faker==40.8.0

### Task 3.2 — Verify the installation

Before using `faker` in a real pipeline, confirm it was installed correctly by:

1. Importing the `Faker` class from the `faker` library.
2. Creating a `Faker` instance.
3. Printing a randomly generated **full name** and **date of birth**.

> 🤖 **Genie Code tip:**
> *"Show me a Python example of generating a random name and date of birth using the Faker library."*

In [0]:
from faker import Faker

fake = Faker()

print(f"Full name: {fake.name()}")
print(f"Date of birth: {fake.date_of_birth(minimum_age=18, maximum_age=85)}")

## Exercise 4: Generate and Analyze Synthetic Patient Data

Now that `faker` is installed, put it to work. Your team needs a synthetic dataset of patient admission records to test a new data pipeline. The data must look realistic enough to validate transformations and aggregations, but must contain **no real patient information** to comply with HIPAA policies.

In this exercise, you generate a synthetic dataset and run a basic analysis on it.

### Task 4.1 — Generate a synthetic patient admissions DataFrame

Use `faker` and the Spark session (`spark`) to create a **Spark DataFrame** with **100 synthetic patient admission records**. Each record must include:

| Column | Description |
|---|---|
| `patient_id` | Integer, 1 to 100 |
| `full_name` | Randomly generated full name |
| `date_of_birth` | Random date between `1940-01-01` and `2005-12-31`, as a string |
| `admission_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `diagnosis_code` | One of: `I21`, `J18`, `E11`, `K80`, `N39` (randomly chosen) |

Display the first 10 rows of the resulting DataFrame.

> 🤖 **Genie Code tip:**
> *"How do I generate a list of Python dictionaries using the Faker library, then create a PySpark DataFrame from that list?"*
>
> **Hint:** Use `faker.date_of_birth()` with `minimum_age` and `maximum_age` parameters, or `faker.date_between()` with `start_date` and `end_date`. Use `random.choice()` for the diagnosis code.

### Task 4.1 — Generate a synthetic patient admissions DataFrame

Use `faker` and the Spark session (`spark`) to create a **Spark DataFrame** with **100 synthetic patient admission records**. Each record must include:

| Column | Description |
|---|---|
| `patient_id` | Integer, 1 to 100 |
| `full_name` | Randomly generated full name |
| `date_of_birth` | Random date between `1940-01-01` and `2005-12-31`, as a string |
| `admission_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `diagnosis_code` | One of: `I21`, `J18`, `E11`, `K80`, `N39` (randomly chosen) |

Display the first 10 rows of the resulting DataFrame.

> 🤖 **Genie Code tip:**
> *"How do I generate a list of Python dictionaries using the Faker library, then create a PySpark DataFrame from that list?"*
>
> **Hint:** Use `faker.date_of_birth()` with `minimum_age` and `maximum_age` parameters, or `faker.date_between()` with `start_date` and `end_date`. Use `random.choice()` for the diagnosis code.

In [0]:
from faker import Faker
from datetime import date
import random

fake = Faker()
diagnosis_codes = ["I21", "J18", "E11", "K80", "N39"]

patient_records = [
    {
        "patient_id": patient_id,
        "full_name": fake.name(),
        "date_of_birth": fake.date_between(start_date="-86y", end_date="-20y").strftime("%Y-%m-%d"),
        "admission_date": fake.date_between(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31)).strftime("%Y-%m-%d"),
        "diagnosis_code": random.choice(diagnosis_codes),
    }
    for patient_id in range(1, 101)
]

patient_admissions_df = spark.createDataFrame(patient_records)
display(patient_admissions_df.limit(10))

### Task 4.2 — Analyze admissions by diagnosis code

The clinical informatics team wants to know which diagnosis codes appear most frequently in the admission data. Using the Spark DataFrame you created in Task 4.1:

1. Count the number of admissions per `diagnosis_code`.
2. Order the results from **most to least** admissions.
3. Display the result.

> 🤖 **Genie Code tip:**
> *"How do I group by a column, count occurrences, and sort the result in descending order using PySpark DataFrame API?"*

In [0]:
from pyspark.sql.functions import desc

admissions_by_diagnosis_df = (
    patient_admissions_df
    .groupBy("diagnosis_code")
    .count()
    .orderBy(desc("count"))
)

display(admissions_by_diagnosis_df)